In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install faiss-cpu -q

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# building the knowledge base - just the correct answer text for each row
print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = embed_model.encode(kb, show_progress_bar=True, batch_size=64)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print("Knowledge base successfully created, size:", len(kb))

# loading models once so we don't reload them for every question
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
bert_tok = AutoTokenizer.from_pretrained('bert-base-uncased')

# ---------- Q1 ----------
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

res_q1 = zs(prompt_150, candidate_labels=labels_150)
score_map_q1 = dict(zip(res_q1['labels'], res_q1['scores']))
q1_score = round(score_map_q1[ans_150], 3)
print("Q1 answer:", q1_score)

# ---------- Q2 ----------
q_embed_150 = embed_model.encode([prompt_150])
distances, retrieved_indices = index.search(q_embed_150, 10)
retrieved_indices = retrieved_indices[0]

# find where the true doc (index 150 in kb) landed
q2_rank = None
for rank, doc_idx in enumerate(retrieved_indices, start=1):
    if doc_idx == 150:
        q2_rank = rank
        break
print("Q2 answer (rank of true doc in FAISS top 10):", q2_rank)

# ---------- Q3 ----------
docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# sort indices by cross encoder score, descending
order = np.argsort(ce_scores)[::-1]
sorted_doc_ids = [retrieved_indices[i] for i in order]

q3_rank = None
for rank, doc_idx in enumerate(sorted_doc_ids, start=1):
    if doc_idx == 150:
        q3_rank = rank
        break
print("Q3 answer (rank after reranking):", q3_rank)

# ---------- Q4 ----------
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
q_embed_42 = embed_model.encode([prompt_42])
_, retrieved_42 = index.search(q_embed_42, 5)
docs_42 = [kb[i] for i in retrieved_42[0]]

concat_docs_42 = " ".join(docs_42)
rag_string_42 = f"Context: {concat_docs_42} Question: {prompt_42}"
tokens_42 = bert_tok(rag_string_42, truncation=False)['input_ids']
print("Q4 answer (token count):", len(tokens_42))

# ---------- Q5 ----------
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"
res_q5 = zs(rag_string_150, candidate_labels=labels_150)
score_map_q5 = dict(zip(res_q5['labels'], res_q5['scores']))
q5_score = round(score_map_q5[ans_150], 3)
print("Q5 answer:", q5_score)

# ---------- Q6 ----------
adversarial_doc = kb[999]
adv_string_150 = f"Context: {adversarial_doc} Question: {prompt_150}"
res_q6 = zs(adv_string_150, candidate_labels=labels_150)
score_map_q6 = dict(zip(res_q6['labels'], res_q6['scores']))
q6_score = round(score_map_q6[ans_150], 3)
print("Q6 answer:", q6_score)

# ---------- Q7 ----------
hits = 0
for i in range(100):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_text = str(row[row['answer']])

    emb_i = embed_model.encode([prompt_i])
    _, top5_idx = index.search(emb_i, 5)
    retrieved_docs = [kb[j] for j in top5_idx[0]]

    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1

hit_rate = round((hits / 100) * 100, 1)
print("Q7 answer (hit rate %):", hit_rate)

# ---------- Q8 ----------
def map_at_3(ranked_letters, correct_letter):
    if correct_letter not in ranked_letters:
        return 0.0
    pos = ranked_letters.index(correct_letter) + 1
    return 1.0 / pos

map_scores = []
for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    opts = {opt: str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']}
    correct_letter = row['answer']

    # retrieve
    emb_i = embed_model.encode([prompt_i])
    _, top5_idx = index.search(emb_i, 5)
    top5_docs = [kb[j] for j in top5_idx[0]]

    # rerank, pick best doc
    ce_pairs = [[prompt_i, doc] for doc in top5_docs]
    ce_out = cross_encoder.predict(ce_pairs)
    best_doc = top5_docs[np.argmax(ce_out)]

    # augment + predict
    rag_str = f"Context: {best_doc} Question: {prompt_i}"
    candidate_labels = list(opts.values())
    res = zs(rag_str, candidate_labels=candidate_labels)

    # map label text back to letter, then rank letters by score
    text_to_letter = {v: k for k, v in opts.items()}
    ranked_letters = [text_to_letter[label] for label in res['labels']]
    top3 = ranked_letters[:3]

    map_scores.append(map_at_3(top3, correct_letter))

q8_final = round(sum(map_scores) / len(map_scores), 3)
print("Q8 answer (avg MAP@3 over 20 rows):", q8_final)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.6 MB/s eta 0:00:00
Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q1 answer: 0.384
Q2 answer (rank of true doc in FAISS top 10): 10
Q3 answer (rank after reranking): 1
Q4 answer (token count): 216
Q5 answer: 0.989
Q6 answer: 0.529
Q7 answer (hit rate %): 73.0
Q8 answer (avg MAP@3 over 20 rows): 0.975


In [3]:
!pip install faiss-cpu -q

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# building the knowledge base - just the correct answer text for each row
print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = embed_model.encode(kb, show_progress_bar=True, batch_size=64)
kb_embeddings = np.ascontiguousarray(kb_embeddings.astype('float32'))  # faiss wants this exact dtype
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print("Knowledge base successfully created, size:", len(kb))

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
bert_tok = AutoTokenizer.from_pretrained('bert-base-uncased')


def get_correct_option_score(text, options_dict, correct_letter):
    """Runs zero shot classification and pulls out the score for the correct option"""
    candidate_labels = list(options_dict.values())
    out = zs(text, candidate_labels=candidate_labels)
    score_map = dict(zip(out['labels'], out['scores']))
    return score_map[options_dict[correct_letter]]


def embed_query(text):
    vec = embed_model.encode([text])
    return np.ascontiguousarray(vec.astype('float32'))


# ---------- Q1 ----------
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
opts_150 = {opt: str(row_150[opt]) for opt in ['A', 'B', 'C', 'D', 'E']}
correct_letter_150 = row_150['answer']

q1_score = round(get_correct_option_score(prompt_150, opts_150, correct_letter_150), 3)
print("Q1 answer:", q1_score)

# ---------- Q2 ----------
q_embed_150 = embed_query(prompt_150)
distances, retrieved_indices = index.search(q_embed_150, 10)
retrieved_indices = retrieved_indices[0]

q2_rank = None
for rank, doc_idx in enumerate(retrieved_indices, start=1):
    if doc_idx == 150:
        q2_rank = rank
        break
print("Q2 answer (rank of true doc in FAISS top 10):", q2_rank)

# ---------- Q3 ----------
docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

order = np.argsort(ce_scores)[::-1]
sorted_doc_ids = [retrieved_indices[i] for i in order]

q3_rank = None
for rank, doc_idx in enumerate(sorted_doc_ids, start=1):
    if doc_idx == 150:
        q3_rank = rank
        break
print("Q3 answer (rank after reranking):", q3_rank)

# ---------- Q4 ----------
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
q_embed_42 = embed_query(prompt_42)
_, retrieved_42 = index.search(q_embed_42, 5)
docs_42 = [kb[i] for i in retrieved_42[0]]

concat_docs_42 = " ".join(docs_42)
rag_string_42 = f"Context: {concat_docs_42} Question: {prompt_42}"
tokens_42 = bert_tok(rag_string_42, truncation=False)['input_ids']
print("Q4 answer (token count):", len(tokens_42))

# ---------- Q5 ----------
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"
q5_score = round(get_correct_option_score(rag_string_150, opts_150, correct_letter_150), 3)
print("Q5 answer:", q5_score)

# ---------- Q6 ----------
adversarial_doc = kb[999]
adv_string_150 = f"Context: {adversarial_doc} Question: {prompt_150}"
q6_score = round(get_correct_option_score(adv_string_150, opts_150, correct_letter_150), 3)
print("Q6 answer:", q6_score)

# ---------- Q7 ----------
hits = 0
for i in range(100):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_text = str(row[row['answer']])

    emb_i = embed_query(prompt_i)
    _, top5_idx = index.search(emb_i, 5)
    retrieved_docs = [kb[j] for j in top5_idx[0]]

    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1

hit_rate = round((hits / 100) * 100, 1)
print("Q7 answer (hit rate %):", hit_rate)

# ---------- Q8 ----------
def map_at_3(ranked_letters, correct_letter):
    if correct_letter not in ranked_letters:
        return 0.0
    pos = ranked_letters.index(correct_letter) + 1
    return 1.0 / pos

map_scores = []
for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    opts = {opt: str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']}
    correct_letter = row['answer']

    emb_i = embed_query(prompt_i)
    _, top5_idx = index.search(emb_i, 5)
    top5_docs = [kb[j] for j in top5_idx[0]]

    ce_pairs = [[prompt_i, doc] for doc in top5_docs]
    ce_out = cross_encoder.predict(ce_pairs)
    best_doc = top5_docs[np.argmax(ce_out)]

    rag_str = f"Context: {best_doc} Question: {prompt_i}"
    letters_in_order = list(opts.keys())
    candidate_labels = list(opts.values())
    res = zs(rag_str, candidate_labels=candidate_labels)

    # map each returned label text back to its letter using position, not a text lookup
    # (avoids issues if two options happen to share identical text)
    label_to_pos = {opts[l]: l for l in letters_in_order}
    ranked_letters = [label_to_pos[label] for label in res['labels']]
    top3 = ranked_letters[:3]

    map_scores.append(map_at_3(top3, correct_letter))

q8_final = round(sum(map_scores) / len(map_scores), 3)
print("Q8 answer (avg MAP@3 over 20 rows):", q8_final)

Creating knowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Knowledge base successfully created, size: 2000


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q1 answer: 0.384
Q2 answer (rank of true doc in FAISS top 10): 10
Q3 answer (rank after reranking): 1
Q4 answer (token count): 216
Q5 answer: 0.989
Q6 answer: 0.529
Q7 answer (hit rate %): 73.0
Q8 answer (avg MAP@3 over 20 rows): 0.975
